In [0]:
# List files in the dataset directory to check Grouping and aggregators

display(
    dbutils.fs.ls('/databricks-datasets/online_retail/')
)

# Load the CSV as a DataFrame
retaildf = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("samplingRatio", "0.5")
    .load("/databricks-datasets/online_retail/data-001")
)

# Display the DataFrame
display(retaildf)
# print schema 
retaildf.printSchema()
#---------------------------------------------------------
# Case-1
# Register the DataFrame as a temporary view for SQL queries
retaildf.createOrReplaceTempView("retaildfview")

# Use SQL to group by Country, and calculate total quantity and invoice value
summary_sql = spark.sql("""
    SELECT
        Country,
        count(InvoiceNo) As total_invoices,
        SUM(Quantity) AS total_quantity,
        round(SUM(Quantity * UnitPrice),2) AS total_value
    FROM retaildfview
    GROUP BY Country
""")

# Display the summary DataFrame
display(summary_sql)
#---------------------------------------------------------

# Case 2 using a dataframe and spark functions

#from pyspark.sql.functions import sum as _sum, round as _round, col

from pyspark.sql import functions as sf

# create a new dataframe
# Note - Aliasing is useful if you want to avoid conflicts with Python's built-in sum and round functions.

summary_df = (
    retaildf.groupBy("Country")
    .agg(
    sf.sum("Quantity").alias("total_quantity"),
        sf.count("InvoiceNo").alias("total_invoices"),
        sf.round(sf.sum(sf.col("Quantity") * sf.col("UnitPrice")), 2).alias("total_value")
    )
)
display(summary_df)

# case 3 convert the date format build the same information by year wise

# from pyspark.sql.functions import to_timestamp, dayofmonth, month, year, hour, minute, col
from pyspark.sql import functions as sf


# Parse the InvoiceDate column to timestamp
retaildf = retaildf.withColumn(
    "InvoiceTimestamp",
    sf.to_timestamp(sf.col("InvoiceDate"), "M/d/yy H:mm")
)

# Extract components
retaildf = (
    retaildf
    .withColumn("dd", sf.dayofmonth(sf.col("InvoiceTimestamp")))
    .withColumn("mm", sf.month(sf.col("InvoiceTimestamp")))
    .withColumn("yr", sf.year(sf.col("InvoiceTimestamp")))
    .withColumn("hh", sf.hour(sf.col("InvoiceTimestamp")))
    .withColumn("minutes", sf.minute(sf.col("InvoiceTimestamp")))
    .groupBy("Country","yr")
    .agg(
    sf.sum("Quantity").alias("total_quantity"),
        sf.count("InvoiceNo").alias("total_invoices"),
        sf.round(sf.sum(sf.col("Quantity") * sf.col("UnitPrice")), 2).alias("total_value")
    )
)
# sort by country and year
display(retaildf.orderBy("Country","yr"))


